In [1]:
"""
Add a 'Component' column to CSV files if it does not already exist.
The column is inserted immediately after the 'Benchmark' column.
"""

import os

input_folder = '../02_raw/csv'

for filename in os.listdir(input_folder):
    if filename.endswith('.csv'):
        filepath = os.path.join(input_folder, filename)

        # Read the file contents
        with open(filepath, 'r', encoding='utf-8') as f:
            lines = f.readlines()

        if not lines:
            continue

        # Header (first line)
        header_line = lines[0].strip()
        # Split into columns (accounting for possible quotes)
        header = [col.strip().strip('"') for col in header_line.split(',')]

        # Check if 'Component' column already exists
        if 'Component' not in header:
            # Find the position of 'Benchmark'
            if 'Benchmark' in header:
                idx = header.index('Benchmark')
                # Insert 'Component' immediately after 'Benchmark'
                header.insert(idx + 1, 'Component')
            else:
                # If for some reason 'Benchmark' is missing, insert at the third position
                header.insert(2, 'Component')

            # Form the new header line
            new_header = ','.join(header)
            lines[0] = new_header + '\n'

            # Overwrite the file with the updated header
            with open(filepath, 'w', encoding='utf-8') as f:
                f.writelines(lines)

            print(f"Added 'Component' to file: {filename}")
        else:
            print(f"File already contains 'Component': {filename}")

File already contains 'Component': LAMMPS_Molecular_Dynamics_Simulator_23Jun2022-Model_20k_Atoms.csv


In [2]:
"""
Combine and clean benchmark CSV files.

This script:
    - Reads all CSV files from the input folder.
    - Removes rows where 'Component' contains 'Mid-Tier', 'Median', or 'Low-Tier'.
    - Filters rows where '# Compatible Public Results' contains numeric or empty values.
    - Drops rows that have a value in 'Loop Time (Average)'.
    - Merges 'Ns Per Day (Average)' and 'ns/day (Average)' into a single 'ns/day (Average)' column.
    - Converts values from 'days/ns (Average)' to 'ns/day (Average)' using inversion.
    - Drops unnecessary columns.
    - Renames specific benchmark names via a mapping.
    - Removes the prefix 'Molecular_Dynamics_Simulator_' from the 'Packet' column.
    - Saves the combined DataFrame to a CSV file.
"""

import pandas as pd
import os
import numpy as np

# Paths
input_folder = '../02_raw/csv/'
output_folder = '../02_raw/'
output_file = os.path.join(output_folder, 'combined_benchmarks.csv')

os.makedirs(output_folder, exist_ok=True)

all_dataframes = []

for filename in os.listdir(input_folder):
    if filename.endswith('.csv'):
        filepath = os.path.join(input_folder, filename)
        df = pd.read_csv(filepath)
        # Remove rows where 'Component' contains Mid-Tier, Median, or Low-Tier
        df = df[~df['Component'].astype(str).str.contains('Mid-Tier|Median|Low-Tier')]
        all_dataframes.append(df)

if all_dataframes:
    combined_df = pd.concat(all_dataframes, ignore_index=True)

    # NEW BLOCK: remove rows with non-numeric and non-empty values in '# Compatible Public Results'
    if '# Compatible Public Results' in combined_df.columns:
        def is_valid_number_or_empty(val):
            if pd.isna(val):
                return True
            if isinstance(val, (int, float)):
                return True
            try:
                float(val)
                return True
            except (ValueError, TypeError):
                return False
        valid_mask = combined_df['# Compatible Public Results'].apply(is_valid_number_or_empty)
        combined_df = combined_df[valid_mask].copy()
        combined_df.reset_index(drop=True, inplace=True)

    # 1. Remove rows that have a value in 'Loop Time (Average)'
    if 'Loop Time (Average)' in combined_df.columns:
        combined_df = combined_df[combined_df['Loop Time (Average)'].isna()].copy()
        combined_df = combined_df.reset_index(drop=True)

    # 2. Merge columns 'Ns Per Day (Average)' and 'ns/day (Average)' into 'ns/day (Average)'
    ns_day_combined = pd.Series([np.nan] * len(combined_df))

    if 'ns/day (Average)' in combined_df.columns:
        ns_day_combined = combined_df['ns/day (Average)'].copy()

    if 'Ns Per Day (Average)' in combined_df.columns:
        mask = ns_day_combined.isna() & combined_df['Ns Per Day (Average)'].notna()
        ns_day_combined[mask] = combined_df.loc[mask, 'Ns Per Day (Average)']

    combined_df['ns/day (Average)'] = ns_day_combined

    # 3. Convert the reciprocal value from 'days/ns (Average)' to 'ns/day (Average)'
    if 'days/ns (Average)' in combined_df.columns:
        mask = combined_df['days/ns (Average)'].notna()
        days_ns_values = pd.to_numeric(combined_df.loc[mask, 'days/ns (Average)'], errors='coerce')
        inverse_values = 1.0 / days_ns_values
        combined_df.loc[mask, 'ns/day (Average)'] = inverse_values

    if 'ns/day (Average)' in combined_df.columns:
        before = len(combined_df)
        combined_df = combined_df.dropna(subset=['ns/day (Average)']).copy()
        combined_df.reset_index(drop=True, inplace=True)
        print(f"Rows removed due to empty ns/day: {before - len(combined_df)}")

    # 4. Drop unnecessary columns
    columns_to_drop = ['Support', 'Instructions Detected']
    if 'Loop Time (Average)' in combined_df.columns:
        columns_to_drop.append('Loop Time (Average)')
    if 'Ns Per Day (Average)' in combined_df.columns:
        columns_to_drop.append('Ns Per Day (Average)')
    if 'days/ns (Average)' in combined_df.columns:
        columns_to_drop.append('days/ns (Average)')
    if columns_to_drop:
        combined_df = combined_df.drop(columns=columns_to_drop, errors='ignore')

    # 5. Transform values in the 'Benchmark' column
    if 'Benchmark' in combined_df.columns:
        benchmark_mapping = {
            'ATPase_Simulation_-_327,506_Atoms': 'ATPase_with_327.506_Atoms',
            'Input_ATPase_with_327,506_Atoms': 'ATPase_with_327.506_Atoms',
            'Input_STMV_with_1,066,628_Atoms': 'STMV_with_1.066.628_Atoms',
            'Implementation_MPI_CPU_-_Input_water_GMX50_bare': 'water_GMX50',
            'Input_water_GMX50_bare': 'water_GMX50',
            'Water_Benchmark': 'water_GMX50'
        }
        combined_df['Benchmark'] = combined_df['Benchmark'].replace(benchmark_mapping)

    # 6. Remove the prefix 'Molecular_Dynamics_Simulator_' from the 'Packet' column
    if 'Packet' in combined_df.columns:
        combined_df['Packet'] = combined_df['Packet'].astype(str).str.replace('Molecular_Dynamics_Simulator_', '', regex=False)

    # Save the result
    combined_df.to_csv(output_file, index=False)

    print(f"Processing completed!")
    print(f"Total files processed: {len(all_dataframes)}")
    print(f"Final number of rows: {len(combined_df)}")
    print(f"Final number of columns: {len(combined_df.columns)}")
    print(f"File saved at: {output_file}")

    print("\nRemaining columns in the table:")
    for i, col in enumerate(combined_df.columns, 1):
        print(f"{i:2}. {col}")

    print(f"\nFirst 3 rows of the result:")
    print(combined_df.head(3))

else:
    print("No CSV files found in the folder to combine.")

Rows removed due to empty ns/day: 0
Processing completed!
Total files processed: 1
Final number of rows: 147
Final number of columns: 7
File saved at: ../02_raw/combined_benchmarks.csv

Remaining columns in the table:
 1. Packet
 2. Benchmark
 3. Component
 4. Percentile Rank
 5. # Compatible Public Results
 6. ns/day (Average)
 7. Intervals

First 3 rows of the result:
             Packet        Benchmark                   Component  \
0  LAMMPS_23Jun2022  Model_20k_Atoms        2 x Intel Xeon 6980P   
1  LAMMPS_23Jun2022  Model_20k_Atoms      AMD EPYC 9965 192-Core   
2  LAMMPS_23Jun2022  Model_20k_Atoms  2 x AMD EPYC 9755 128-Core   

  Percentile Rank # Compatible Public Results  ns/day (Average)  Intervals  
0           100th                          27              93.0        1.0  
1            99th                          11              81.0       11.0  
2            98th                          20              78.0        7.0  
